# Transformer Sentence Embeddings — Kaggle Training

## Before running
1. **Add your project as a dataset**: Upload your project folder (containing `train.py`, `data.py`, `models/`, `losses/`) as a Kaggle dataset, then attach it to this notebook.
2. **Set `PROJECT_DIR`** in Cell 2 to the dataset path (e.g. `/kaggle/input/transformer-sbert`).
3. **Enable GPU**: Settings → Accelerator → GPU T4
4. **Enable Internet**: Settings → Internet → On (needed for data download)
5. **Run all cells** top-to-bottom.

In [2]:
!gdown "https://drive.google.com/uc?id=1_NyFvHSSFT1GjEaCGZOWNRawya2wwBmz"
!unzip "transformer - Copy.zip"
# https://drive.google.com/file/d/1_NyFvHSSFT1GjEaCGZOWNRawya2wwBmz/view?usp=sharing

Downloading...
From: https://drive.google.com/uc?id=1_NyFvHSSFT1GjEaCGZOWNRawya2wwBmz
To: /kaggle/working/transformer - Copy.zip
100%|███████████████████████████████████████| 2.02M/2.02M [00:00<00:00, 182MB/s]
Archive:  transformer - Copy.zip
   creating: transformer - Copy/
  inflating: transformer - Copy/clean_data.py  
  inflating: transformer - Copy/data.py  
  inflating: transformer - Copy/evaluate_search.py  
  inflating: transformer - Copy/kaggle_running.ipynb  
  inflating: transformer - Copy/local_running.ipynb  
   creating: transformer - Copy/losses/
  inflating: transformer - Copy/losses/mnr_loss.py  
  inflating: transformer - Copy/losses/triplet_loss.py  
  inflating: transformer - Copy/losses/__init__.py  
   creating: transformer - Copy/losses/__pycache__/
  inflating: transformer - Copy/losses/__pycache__/mnr_loss.cpython-314.pyc  
  inflating: transformer - Copy/losses/__pycache__/triplet_loss.cpython-314.pyc  
  inflating: transformer - Copy/losses/__pycache__/__init

In [3]:
!pip install -q datasets scipy tqdm

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [1]:
import shutil
import os

working_dir = "/kaggle/working"

for item in os.listdir(working_dir):
    path = os.path.join(working_dir, item)

    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    else:
        try:
            os.remove(path)
        except:
            pass

In [4]:
import os, sys, shutil
# transformer - Copy.zip
# ── Set this to your Kaggle dataset path ──────────────────────────────────────
PROJECT_DIR = '/kaggle/input/datasets/amreisa9/sts-222'   # <-- update to your dataset slug
# ─────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working'

if os.path.isdir(PROJECT_DIR):
    print(f'Copying project files from {PROJECT_DIR} ...')
    for item in os.listdir(PROJECT_DIR):
        src = os.path.join(PROJECT_DIR, item)
        dst = os.path.join(WORKING_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print('Done.')
else:
    print(f'PROJECT_DIR not found: {PROJECT_DIR}')
    print('Running from current directory (files must already be present).')

os.makedirs('data', exist_ok=True)
sys.path.insert(0, WORKING_DIR)
print(f'sys.path[0] = {sys.path[0]}')
print('Files in working dir:', [f for f in os.listdir(WORKING_DIR) if not f.startswith('.')][:20])

Copying project files from /kaggle/input/datasets/amreisa9/sts-222 ...
Done.
sys.path[0] = /kaggle/working
Files in working dir: ['stsb_validation.csv', 'data', 'stsb_test.csv', 'transformer - Copy.zip', 'stsb_train.csv', 'transformer - Copy']


In [66]:
# # ── Download STS-B dataset ────────────────────────────────────────────────────
# import pandas as pd

# try:
#     from datasets import load_dataset
#     splits = [
#         ('train',      'data/train.csv'),
#         ('validation', 'data/val.csv'),
#         ('test',       'data/test.csv'),
#     ]
#     for split, out in splits:
#         if os.path.exists(out):
#             print(f'{out} already exists, skipping.')
#             continue
#         df = load_dataset('mteb/stsbenchmark-sts', split=split).to_pandas()
#         if 'score' not in df.columns and 'similarity_score' in df.columns:
#             df = df.rename(columns={'similarity_score': 'score'})
#         df[['sentence1', 'sentence2', 'score']].to_csv(out, index=False)
#         lo, hi = df['score'].min(), df['score'].max()
#         print(f'{out}: {len(df):,} rows  score=[{lo:.1f}, {hi:.1f}]')
#     print('Data ready.')
# except Exception as e:
#     print(f'Download failed: {e}')
#     print('Manually place stsb_train/validation/test.csv in data/')

data/stsb_train.csv: 5,749 rows  score=[0.0, 5.0]
data/stsb_validation.csv: 1,500 rows  score=[0.0, 5.0]
data/stsb_test.csv: 1,379 rows  score=[0.0, 5.0]
Data ready.


In [5]:
import shutil, os

os.makedirs("data", exist_ok=True)

shutil.copy("stsb_train.csv", "data/stsb_train.csv")
shutil.copy("stsb_validation.csv", "data/stsb_validation.csv")
shutil.copy("stsb_test.csv", "data/stsb_test.csv")

'data/stsb_test.csv'

In [6]:
!head -n 5 data/stsb_train.csv

sentence1,sentence2,score
Ann older man with a gray beard is cooking in the kitchen,Older man with beard is cooking,0.666193821907315
Ann older man with a gray beard is cooking in the kitchen,The man is cooking for his wide,0.36295557621867963
Ann older man with a gray beard is cooking in the kitchen,Older woman with beard is cooking,0.6086726757178349
A man in an apron smiling as he fries food,A woman is baking cookies,0.1548350667974263


In [7]:
# ── Train ─────────────────────────────────────────────────────────────────────
# Runs train.py as-is — uses CONFIG defined inside train.py
!python "/kaggle/working/transformer - Copy/train.py"

Device      : cuda
Loss        : MNR  temperature=0.15
d_model=256  n_layers=4  n_heads=4  d_ff=512

Loading data ...
Vocab size    : 85,415  (saved to vocab.pkl)
Train pairs   : 100,995
Val STS pairs : 19,657
Steps/epoch   : 1579   total: 31580
Parameters    : 23,974,656

Starting training from scratch
Epoch 01/20  lr=2.98e-04  | MNR loss=0.8294  | Val Spearman=0.6012              
  * Best model saved  (val Spearman = 0.6012)
Epoch 02/20  lr=2.93e-04  | MNR loss=0.5743  | Val Spearman=0.6327              
  * Best model saved  (val Spearman = 0.6327)
Epoch 03/20  lr=2.84e-04  | MNR loss=0.4879  | Val Spearman=0.6350              
  * Best model saved  (val Spearman = 0.6350)
Epoch 04/20  lr=2.72e-04  | MNR loss=0.4332  | Val Spearman=0.6396              
  * Best model saved  (val Spearman = 0.6396)
Epoch 05/20  lr=2.57e-04  | MNR loss=0.3956  | Val Spearman=0.6346              
Epoch 06/20  lr=2.39e-04  | MNR loss=0.3654  | Val Spearman=0.6433              
  * Best model saved  (va

In [8]:
# ── Evaluate (semantic search metrics) ───────────────────────────────────────
# Requires best_model.pt to exist (produced by Cell 4)
!python "/kaggle/working/transformer - Copy/evaluate_search.py"

Mining hard negatives — encoding 1139276 sentences ...
Mining: 100%|█████████████████████████████| 51894/51894 [59:28<00:00, 14.54it/s]
Mined 297,760 hard-negative triplets
Saved 297,760 hard-negative triplets to data/train_hard_negatives.csv


In [12]:
for root, dirs, files in os.walk("/kaggle/working/transformer - Copy"):
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

/kaggle/working/transformer - Copy/data.py
/kaggle/working/transformer - Copy/model.py
/kaggle/working/transformer - Copy/evaluate_search.py
/kaggle/working/transformer - Copy/clean_data.py
/kaggle/working/transformer - Copy/train.py
/kaggle/working/transformer - Copy/train_triplet.py
/kaggle/working/transformer - Copy/models/blocks/encoder_layer.py
/kaggle/working/transformer - Copy/models/embedding/positional_encoding.py
/kaggle/working/transformer - Copy/models/embedding/transformer_embedding.py
/kaggle/working/transformer - Copy/models/layers/multi_head_attention.py
/kaggle/working/transformer - Copy/models/layers/scale_dot_product_attention.py
/kaggle/working/transformer - Copy/models/layers/position_wise_feed_forward.py
/kaggle/working/transformer - Copy/models/model/transformer.py
/kaggle/working/transformer - Copy/models/model/sts_transformer.py
/kaggle/working/transformer - Copy/models/model/encoder.py
/kaggle/working/transformer - Copy/losses/__init__.py
/kaggle/working/trans

In [21]:
# ── Evaluate final model (Recall@K, MRR, Spearman) ───────────────────────────
import pickle, torch
import sys

sys.path.append('/kaggle/working/transformer - Copy')

from evaluate_search import evaluate

with open('vocab.pkl', 'rb') as _vf:
    vocab = pickle.load(_vf)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EVAL_CONFIG = {
    'd_model':       256,
    'n_layers':      4,
    'n_heads':       4,
    'd_ff':          512,
    'max_len':       128,
    'pooling':       'mean',
    'pos_threshold': 0.65,
}

print('=== MNR model (best_model.pt) ===')
evaluate('best_model.pt', 'data/stsb_validation.csv', vocab, EVAL_CONFIG, DEVICE)

print('\n=== Hard-tuned model (final_hard_tuned_model.pt) ===')
evaluate('final_hard_tuned_model.pt', 'data/stsb_validation.csv', vocab, EVAL_CONFIG, DEVICE)

=== MNR model (best_model.pt) ===
Encoding 26094 corpus sentences ...

Semantic Search Evaluation on data/stsb_validation.csv
  Recall@1   : 0.3277   (32.8%)
  Recall@5   : 0.6330   (63.3%)
  Recall@10  : 0.6931   (69.3%)
  MRR        : 0.4670
  Spearman ρ : 0.6433

=== Hard-tuned model (final_hard_tuned_model.pt) ===
Encoding 26094 corpus sentences ...

Semantic Search Evaluation on data/stsb_validation.csv
  Recall@1   : 0.1445   (14.4%)
  Recall@5   : 0.2573   (25.7%)
  Recall@10  : 0.2874   (28.7%)
  MRR        : 0.1985
  Spearman ρ : 0.4764


{'recall@1': 0.14447666754547853,
 'recall@5': 0.25731610862114423,
 'recall@10': 0.2873714737674664,
 'mrr': 0.19852808616623968,
 'spearman': np.float64(0.47639210478805954)}

In [26]:
# ── Inference demo ────────────────────────────────────────────────────────────
import os, pickle
import torch
import torch.nn.functional as F

from models.model.transformer import Transformer

CONFIG = {
    'max_len':  128,
    'd_model':  256,
    'n_layers': 4,
    'n_heads':  4,
    'd_ff':     512,
    'pooling':  'mean',
    # Use the hard-tuned model if available, otherwise fall back to MNR model
    'save_path': 'best_model.pt',
}

with open('vocab.pkl', 'rb') as _vf:
    vocab = pickle.load(_vf)
print(f'Vocab loaded from vocab.pkl ({len(vocab):,} tokens)')

model = Transformer(
    vocab_size=len(vocab),
    d_model=CONFIG['d_model'],
    n_layers=CONFIG['n_layers'],
    n_heads=CONFIG['n_heads'],
    d_ff=CONFIG['d_ff'],
    max_len=CONFIG['max_len'],
    dropout=0.0,
    pooling=CONFIG['pooling'],
).to(DEVICE)

model.load_state_dict(torch.load(CONFIG['save_path'], map_location=DEVICE))
model.eval()
print(f"Model loaded from {CONFIG['save_path']}")

Vocab loaded from vocab.pkl (85,415 tokens)
Model loaded from best_model.pt


In [27]:
@torch.no_grad()
def embed(text):
    cls_id = vocab.word2idx[vocab.CLS_TOKEN]
    sep_id = vocab.word2idx[vocab.SEP_TOKEN]
    ids = torch.tensor(
        [[cls_id] + vocab.encode(text)[:CONFIG['max_len'] - 2] + [sep_id]],
        dtype=torch.long
    ).to(DEVICE)
    return model.encode(ids, normalize=True)   # (1, d_model) unit vector


def similarity(t1, t2):
    return (embed(t1) * embed(t2)).sum().item()


test_pairs = [
    ('A dog is running in the park',   'A dog is playing outside'),
    ('The man is eating a sandwich',   'A person is having lunch'),
    ('A woman is singing a song',      'A lady is performing music'),
    ('The cat sat on the mat',         'A dog is running in the park'),
    ('A man is riding a bicycle',      'Someone is driving a car'),
    ('I love pizza',                   'The stock market crashed today'),
]

print(f'{"Sentence 1":<42} {"Sentence 2":<42} {"Sim":>5}')
print('-' * 95)
for s1, s2 in test_pairs:
    sim = similarity(s1, s2)
    bar = '#' * int((sim + 1) * 10)
    print(f'{s1:<42} {s2:<42} {sim:>5.3f}  {bar}')

print(CONFIG['save_path'])


Sentence 1                                 Sentence 2                                   Sim
-----------------------------------------------------------------------------------------------
A dog is running in the park               A dog is playing outside                   0.480  ##############
The man is eating a sandwich               A person is having lunch                   0.309  #############
A woman is singing a song                  A lady is performing music                 0.552  ###############
The cat sat on the mat                     A dog is running in the park               0.331  #############
A man is riding a bicycle                  Someone is driving a car                   0.355  #############
I love pizza                               The stock market crashed today             0.015  ##########
best_model.pt
